# 18vD — Pre-holdout final fits and blind holdout/June predictions

This stage refits only the development-selected baseline,
Gaussian-process and tree candidates on their model-specific
pre-holdout fitting memberships.

It then generates blind predictive residual and HKO-temperature
quantiles for:

- the 40 locked internal-holdout date–decision rows;
- the 119 June external-test date–decision rows.

The June predictions use the identical pre-holdout fitted
estimator as the internal holdout. There is no refit on May
holdout labels.

The output contains no realised HKO value, event outcome, realised
residual, market probability or market price. Scoring is deferred
to the subsequent event-probability and evaluation stage.

**Revision v2.** The model-specific holdout freeze is read from the final-fit membership and attached to each blind evaluation row; the evaluation-support file is not assumed to duplicate that field.

The notebook explicitly verifies that internal-holdout predictions use `PRE_HOLDOUT_FREEZE` and June predictions use `IDENTICAL_PRE_HOLDOUT_FIT`.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    RBF,
    WhiteKernel,
)
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / ".git").exists():
    raise RuntimeError(
        f"Run this notebook from the repository root, not {ROOT}"
    )

UTC = timezone.utc
STEP = "18vD"
RANDOM_SEED = 20260721
SAMPLE_ORIGIN = pd.Timestamp("2026-03-16")

RULES = [
    "24h_prior",
    "12h_prior",
    "6h_prior",
    "event_day_open",
]

A_DIR = (
    ROOT
    / "data/processed/18vA_residual_model_design_and_features"
)
C_DIR = (
    ROOT
    / "data/processed/18vC_common_support_scoring_and_selection"
)
U_DIR = (
    ROOT
    / "data/processed/18uB_model_specific_freeze_support"
)

CANDIDATE_PATH = A_DIR / "18vA_candidate_registry.csv"
QUANTILE_PATH = A_DIR / "18vA_quantile_grid.csv"
BLIND_FEATURE_PATH = (
    A_DIR / "18vA_blind_evaluation_feature_panel.csv"
)
A_SUMMARY_PATH = A_DIR / "18vA_summary.json"
A_MANIFEST_PATH = A_DIR / "18vA_sha256_manifest.csv"

SELECTION_PATH = C_DIR / "18vC_selected_candidates.json"
SCORE_PATH = C_DIR / "18vC_candidate_development_scores.csv"
C_SUMMARY_PATH = C_DIR / "18vC_summary.json"
C_MANIFEST_PATH = C_DIR / "18vC_sha256_manifest.csv"

FINAL_FIT_PATH = U_DIR / "18uB_final_fit_membership.csv"
EVALUATION_PATH = U_DIR / "18uB_evaluation_support.csv"
U_SUMMARY_PATH = U_DIR / "18uB_summary.json"
U_MANIFEST_PATH = U_DIR / "18uB_sha256_manifest.csv"

OUT_DIR = (
    ROOT
    / "data/processed/18vD_selected_models_blind_predictions"
)
REPORT_DIR = (
    ROOT
    / "reports/18vD_selected_models_blind_predictions"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_INPUTS = [
    CANDIDATE_PATH,
    QUANTILE_PATH,
    BLIND_FEATURE_PATH,
    A_SUMMARY_PATH,
    A_MANIFEST_PATH,
    SELECTION_PATH,
    SCORE_PATH,
    C_SUMMARY_PATH,
    C_MANIFEST_PATH,
    FINAL_FIT_PATH,
    EVALUATION_PATH,
    U_SUMMARY_PATH,
    U_MANIFEST_PATH,
]

for path in REQUIRED_INPUTS:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required verified input is missing: {path}"
        )

TREE_QUANTILES = np.array(
    [0.05, 0.25, 0.50, 0.75, 0.95],
    dtype=float,
)
TREE_POOLED_FEATURES = [
    "day_index",
    "forecast_daily_max_c",
    "decision_rule_order",
    "forecast_lead_hours",
    "run_initialisation_hour_utc",
    "day_of_year_sin",
    "day_of_year_cos",
]
TREE_RULE_FEATURES = [
    "day_index",
    "forecast_daily_max_c",
    "forecast_lead_hours",
    "run_initialisation_hour_utc",
    "day_of_year_sin",
    "day_of_year_cos",
]

In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


def parse_bool(
    series: pd.Series,
    *,
    name: str,
) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    parsed = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            }
        )
    )

    if parsed.isna().any():
        bad = series.loc[
            parsed.isna()
        ].drop_duplicates().tolist()
        raise ValueError(
            f"Could not parse Boolean column {name}: {bad}"
        )

    return parsed.astype(bool)


def verify_manifest(path: Path) -> None:
    manifest = pd.read_csv(path)
    failures: list[str] = []

    for row in manifest.itertuples(index=False):
        candidate = ROOT / row.path

        if not candidate.is_file():
            failures.append(f"MISSING: {row.path}")
            continue

        if sha256_file(candidate) != row.sha256:
            failures.append(f"HASH: {row.path}")

        if candidate.stat().st_size != int(row.size_bytes):
            failures.append(f"SIZE: {row.path}")

    if failures:
        raise AssertionError(
            f"Manifest verification failed for {path}:\n"
            + "\n".join(failures)
        )


def weighted_quantile(
    values: np.ndarray,
    weights: np.ndarray,
    levels: np.ndarray,
) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    levels = np.asarray(levels, dtype=float)

    order = np.argsort(values)
    values = values[order]
    weights = weights[order]

    if (
        len(values) == 0
        or not np.isfinite(values).all()
        or not np.isfinite(weights).all()
        or np.any(weights < 0)
        or weights.sum() <= 0
    ):
        raise ValueError(
            "Invalid weighted-quantile input."
        )

    cumulative = (
        np.cumsum(weights) - 0.5 * weights
    ) / weights.sum()

    return np.interp(
        levels,
        cumulative,
        values,
        left=values[0],
        right=values[-1],
    )


def add_engineered_features(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    output = frame.copy()
    output["event_date"] = pd.to_datetime(
        output["event_date"],
        errors="raise",
    )

    for column in [
        "decision_cutoff_utc",
        "selected_run_initialisation_utc",
        "selected_run_available_utc",
    ]:
        if column in output.columns:
            output[column] = pd.to_datetime(
                output[column],
                utc=True,
                errors="raise",
            )

    if "day_index" not in output.columns:
        output["day_index"] = (
            output["event_date"] - SAMPLE_ORIGIN
        ).dt.days.astype(float)

    if "forecast_lead_hours" not in output.columns:
        output["forecast_lead_hours"] = (
            output["decision_cutoff_utc"]
            - output["selected_run_available_utc"]
        ).dt.total_seconds() / 3600.0

    if (
        "run_initialisation_hour_utc"
        not in output.columns
    ):
        output[
            "run_initialisation_hour_utc"
        ] = output[
            "selected_run_initialisation_utc"
        ].dt.hour.astype(float)

    day_of_year = output[
        "event_date"
    ].dt.dayofyear.astype(float)

    if "day_of_year_sin" not in output.columns:
        output["day_of_year_sin"] = np.sin(
            2.0 * np.pi * day_of_year / 366.0
        )
    if "day_of_year_cos" not in output.columns:
        output["day_of_year_cos"] = np.cos(
            2.0 * np.pi * day_of_year / 366.0
        )

    return output


def interpolate_tree_quantiles(
    trained: np.ndarray,
    output_levels: np.ndarray,
) -> np.ndarray:
    trained = np.maximum.accumulate(
        np.asarray(trained, dtype=float),
        axis=1,
    )

    output = np.vstack(
        [
            np.interp(
                output_levels,
                TREE_QUANTILES,
                row,
                left=row[0],
                right=row[-1],
            )
            for row in trained
        ]
    )

    return np.maximum.accumulate(
        output,
        axis=1,
    )


for manifest_path in [
    A_MANIFEST_PATH,
    C_MANIFEST_PATH,
    U_MANIFEST_PATH,
]:
    verify_manifest(manifest_path)

summaries = {}
for name, path in [
    ("18vA", A_SUMMARY_PATH),
    ("18vC", C_SUMMARY_PATH),
    ("18uB", U_SUMMARY_PATH),
]:
    with path.open(encoding="utf-8") as handle:
        summaries[name] = json.load(handle)
    if summaries[name].get("verdict") != "PASS":
        raise AssertionError(
            f"{name} is not a PASS release."
        )

with SELECTION_PATH.open(encoding="utf-8") as handle:
    selection = json.load(handle)

candidates = pd.read_csv(
    CANDIDATE_PATH,
    low_memory=False,
)
scores = pd.read_csv(
    SCORE_PATH,
    low_memory=False,
)
quantile_grid = pd.read_csv(
    QUANTILE_PATH,
    low_memory=False,
)
blind = pd.read_csv(
    BLIND_FEATURE_PATH,
    low_memory=False,
)
final_fit = pd.read_csv(
    FINAL_FIT_PATH,
    low_memory=False,
)
evaluation = pd.read_csv(
    EVALUATION_PATH,
    low_memory=False,
)

for frame in [
    blind,
    final_fit,
    evaluation,
]:
    frame["event_date"] = pd.to_datetime(
        frame["event_date"],
        errors="raise",
    )

for frame in [
    blind,
    final_fit,
    evaluation,
]:
    for column in [
        "decision_cutoff_utc",
        "selected_run_initialisation_utc",
        "selected_run_available_utc",
        "model_specific_holdout_freeze_utc",
    ]:
        if column in frame.columns:
            frame[column] = pd.to_datetime(
                frame[column],
                utc=True,
                errors="raise",
            )

for column in [
    "contains_holdout_label",
    "contains_external_label",
]:
    final_fit[column] = parse_bool(
        final_fit[column],
        name=f"final_fit.{column}",
    )

evaluation[
    "refit_on_holdout_labels"
] = parse_bool(
    evaluation[
        "refit_on_holdout_labels"
    ],
    name="evaluation.refit_on_holdout_labels",
)

expected_parameter_sources = {
    "LOCKED_INTERNAL_HOLDOUT": "PRE_HOLDOUT_FREEZE",
    "EXTERNAL_OOT_TRANSFER": "IDENTICAL_PRE_HOLDOUT_FIT",
}

for evaluation_stage, expected_source in expected_parameter_sources.items():
    stage_sources = set(
        evaluation.loc[
            evaluation["evaluation_stage"].eq(
                evaluation_stage
            ),
            "fitted_parameter_source",
        ].dropna().astype(str)
    )

    if stage_sources != {expected_source}:
        raise AssertionError(
            f"{evaluation_stage}: expected fitted-parameter source "
            f"{expected_source}, found {sorted(stage_sources)}."
        )

final_fit = add_engineered_features(
    final_fit
)
blind = add_engineered_features(
    blind
)

quantile_levels = quantile_grid[
    "quantile_level"
].to_numpy(dtype=float)
residual_columns = quantile_grid[
    "residual_quantile_column"
].tolist()
hko_columns = quantile_grid[
    "hko_quantile_column"
].tolist()

role_columns = [
    "selected_baseline",
    "selected_gaussian_process",
    "selected_tree",
    "selected_overall",
]

selected_role_rows = []
for role in role_columns:
    selected_role_rows.append(
        {
            "selection_role": role,
            "candidate_id": selection[role],
        }
    )
role_table = pd.DataFrame(
    selected_role_rows
)

unique_selected = (
    role_table.groupby(
        "candidate_id",
        as_index=False,
    )
    .agg(
        selection_roles=(
            "selection_role",
            lambda values: "|".join(
                sorted(values)
            ),
        )
    )
    .merge(
        candidates,
        on="candidate_id",
        how="left",
        validate="one_to_one",
    )
)

if len(unique_selected) != 3:
    raise AssertionError(
        f"Expected three distinct family winners, "
        f"found {len(unique_selected)}."
    )

if set(
    unique_selected["model_family"]
) != {
    "BASELINE",
    "GAUSSIAN_PROCESS",
    "TREE",
}:
    raise AssertionError(
        "The selected set does not contain one baseline, "
        "one GP and one tree candidate."
    )

forbidden_blind = {
    "hko_daily_max_c",
    "residual_c",
    "forecast_error_c",
    "Y_event_int",
    "Y_no_int",
    "p_market",
    "market_binary_brier",
    "market_binary_log_score",
    "current_label_available_utc",
}

present = forbidden_blind.intersection(
    blind.columns
)
if present:
    raise AssertionError(
        "Blind feature panel contains outcomes or market "
        f"fields: {sorted(present)}"
    )

if len(blind) != 159:
    raise AssertionError(
        f"Expected 159 blind rows, found {len(blind)}."
    )

if final_fit[
    [
        "contains_holdout_label",
        "contains_external_label",
    ]
].any().any():
    raise AssertionError(
        "Holdout or external labels enter final fitting."
    )

print("Verified selected candidates and blind supports: PASS")
display(unique_selected)

Verified selected candidates and blind supports: PASS


,candidate_id,selection_roles,model_family,scope_type,scope_id_pattern,distribution_type,complexity_rank,principal_family_comparison,candidate_order,uses_market_information,uses_artificial_ensemble_features,uses_fixed_gaussian_bridge,development_only_selection,holdout_or_external_used_for_selection
0,catboost_quantile_pooled,selected_tree,TREE,POOLED,pooled_all_rules,INTERPOLATED_QUANTILES,8,True,8,False,False,False,True,False
1,gp_matern32_rule,selected_gaussian_process,GAUSSIAN_PROCESS,RULE_SPECIFIC,rule_specific_{decision_rule},GAUSSIAN_QUANTILES,7,True,7,False,False,False,True,False
2,pooled_empirical_residual,selected_baseline|selected_overall,BASELINE,POOLED,pooled_all_rules,EMPIRICAL_QUANTILES,4,False,4,False,False,False,True,False


In [3]:
key = ["event_date", "decision_rule"]

def selected_scope_ids(
    candidate: dict[str, Any],
) -> list[str]:
    if candidate["scope_type"] == "POOLED":
        return ["pooled_all_rules"]

    return [
        f"rule_specific_{rule}"
        for rule in RULES
    ]


prediction_frames = []
diagnostic_rows = []
alpha_text = ",".join(
    f"{alpha:.2f}"
    for alpha in TREE_QUANTILES
)

for candidate in unique_selected.to_dict(
    orient="records"
):
    candidate_id = candidate["candidate_id"]

    for scope_id in selected_scope_ids(candidate):
        applicable_rule = (
            None
            if scope_id == "pooled_all_rules"
            else scope_id.replace(
                "rule_specific_",
                "",
                1,
            )
        )

        train = final_fit.loc[
            final_fit["scope_id"].eq(
                scope_id
            )
        ].copy()

        support = evaluation.loc[
            evaluation["scope_id"].eq(
                scope_id
            )
        ][
            [
                "event_date",
                "decision_rule",
                "evaluation_stage",
                "scope_id",
                "fitted_parameter_source",
                "refit_on_holdout_labels",
            ]
        ].copy()

        freeze_values = train[
            "model_specific_holdout_freeze_utc"
        ].dropna().drop_duplicates()

        if len(freeze_values) != 1:
            raise AssertionError(
                f"Expected one holdout freeze for {scope_id}, "
                f"found {freeze_values.tolist()}."
            )

        support[
            "model_specific_holdout_freeze_utc"
        ] = freeze_values.iloc[0]

        evaluate = support.merge(
            blind,
            on=key,
            how="left",
            validate="one_to_one",
        )

        if applicable_rule is not None:
            if not train[
                "decision_rule"
            ].eq(applicable_rule).all():
                raise AssertionError(
                    "Rule-specific fit contains another rule."
                )
            if not evaluate[
                "decision_rule"
            ].eq(applicable_rule).all():
                raise AssertionError(
                    "Rule-specific evaluation contains "
                    "another rule."
                )

        if evaluate[
            [
                "forecast_daily_max_c",
                "day_index",
                "forecast_lead_hours",
                "run_initialisation_hour_utc",
                "day_of_year_sin",
                "day_of_year_cos",
            ]
        ].isna().any().any():
            raise AssertionError(
                f"Blind feature join failed for {scope_id}."
            )

        y_train = train[
            "residual_c"
        ].to_numpy(dtype=float)
        weights = train[
            "date_balanced_fit_weight"
        ].to_numpy(dtype=float)

        if candidate_id == "raw_deterministic":
            quantiles = np.zeros(
                (
                    len(evaluate),
                    len(quantile_levels),
                )
            )
            parameters = {
                "residual_shift_c": 0.0
            }

        elif candidate_id in {
            "pooled_mean_residual",
            "rule_mean_residual",
        }:
            mean_residual = float(
                np.average(
                    y_train,
                    weights=weights,
                )
            )
            quantiles = np.full(
                (
                    len(evaluate),
                    len(quantile_levels),
                ),
                mean_residual,
            )
            parameters = {
                "weighted_mean_residual_c": (
                    mean_residual
                )
            }

        elif candidate_id in {
            "pooled_empirical_residual",
            "rule_empirical_residual",
        }:
            empirical = weighted_quantile(
                y_train,
                weights,
                quantile_levels,
            )
            quantiles = np.tile(
                empirical,
                (len(evaluate), 1),
            )
            parameters = {
                "weighted_mean_residual_c": float(
                    np.average(
                        y_train,
                        weights=weights,
                    )
                ),
                "minimum_residual_c": float(
                    y_train.min()
                ),
                "maximum_residual_c": float(
                    y_train.max()
                ),
            }

        elif candidate_id in {
            "gp_rbf_rule",
            "gp_matern32_rule",
        }:
            if applicable_rule is None:
                raise AssertionError(
                    "Selected GP is not rule-specific."
                )

            scaler = StandardScaler()
            x_train = scaler.fit_transform(
                train[["day_index"]].to_numpy(
                    dtype=float
                )
            )
            x_evaluate = scaler.transform(
                evaluate[
                    ["day_index"]
                ].to_numpy(dtype=float)
            )

            if candidate_id == "gp_rbf_rule":
                smooth = RBF(
                    length_scale=1.0,
                    length_scale_bounds=(
                        0.05,
                        10.0,
                    ),
                )
            else:
                smooth = Matern(
                    length_scale=1.0,
                    length_scale_bounds=(
                        0.05,
                        10.0,
                    ),
                    nu=1.5,
                )

            kernel = (
                ConstantKernel(
                    1.0,
                    constant_value_bounds=(
                        1e-2,
                        1e2,
                    ),
                )
                * smooth
                + WhiteKernel(
                    noise_level=0.25,
                    noise_level_bounds=(
                        1e-3,
                        10.0,
                    ),
                )
            )

            model = GaussianProcessRegressor(
                kernel=kernel,
                alpha=1e-8,
                normalize_y=True,
                n_restarts_optimizer=0,
                random_state=RANDOM_SEED,
            )

            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model.fit(
                    x_train,
                    y_train,
                )

            mean, standard_deviation = model.predict(
                x_evaluate,
                return_std=True,
            )
            standard_deviation = np.maximum(
                standard_deviation,
                1e-6,
            )
            quantiles = (
                mean[:, None]
                + standard_deviation[:, None]
                * norm.ppf(
                    quantile_levels
                )[None, :]
            )
            parameters = {
                "fitted_kernel": str(
                    model.kernel_
                ),
                "log_marginal_likelihood": float(
                    model.log_marginal_likelihood_value_
                ),
                "day_index_mean": float(
                    scaler.mean_[0]
                ),
                "day_index_scale": float(
                    scaler.scale_[0]
                ),
            }

        elif candidate_id in {
            "catboost_quantile_pooled",
            "catboost_quantile_rule",
        }:
            feature_columns = (
                TREE_POOLED_FEATURES
                if candidate_id
                == "catboost_quantile_pooled"
                else TREE_RULE_FEATURES
            )

            model = CatBoostRegressor(
                loss_function=(
                    "MultiQuantile:alpha="
                    + alpha_text
                ),
                iterations=250,
                depth=3,
                learning_rate=0.03,
                l2_leaf_reg=8.0,
                random_seed=RANDOM_SEED,
                bootstrap_type="No",
                random_strength=0.0,
                verbose=False,
                allow_writing_files=False,
                thread_count=1,
            )
            model.fit(
                train[
                    feature_columns
                ].to_numpy(dtype=float),
                y_train,
                sample_weight=weights,
                verbose=False,
            )

            trained = np.asarray(
                model.predict(
                    evaluate[
                        feature_columns
                    ].to_numpy(dtype=float)
                ),
                dtype=float,
            )

            if trained.ndim == 1:
                trained = trained[:, None]

            if trained.shape != (
                len(evaluate),
                len(TREE_QUANTILES),
            ):
                raise AssertionError(
                    "Unexpected MultiQuantile shape: "
                    f"{trained.shape}"
                )

            quantiles = interpolate_tree_quantiles(
                trained,
                quantile_levels,
            )
            importance = (
                model.get_feature_importance()
            )
            parameters = {
                "loss": "MultiQuantile",
                "training_quantiles": (
                    TREE_QUANTILES.tolist()
                ),
                "iterations": 250,
                "depth": 3,
                "learning_rate": 0.03,
                "l2_leaf_reg": 8.0,
                "features": feature_columns,
                "feature_importance": {
                    feature: float(value)
                    for feature, value in zip(
                        feature_columns,
                        importance,
                    )
                },
            }

        else:
            raise ValueError(
                f"Unsupported selected candidate: "
                f"{candidate_id}"
            )

        quantiles = np.maximum.accumulate(
            np.asarray(
                quantiles,
                dtype=float,
            ),
            axis=1,
        )

        output = evaluate[
            [
                "event_date",
                "decision_rule",
                "decision_rule_order",
                "decision_cutoff_utc",
                "evaluation_block",
                "evaluation_stage",
                "forecast_daily_max_c",
                "selected_run_key",
                "selected_run_initialisation_utc",
                "selected_run_available_utc",
                "scope_id",
                "fitted_parameter_source",
                "refit_on_holdout_labels",
                "model_specific_holdout_freeze_utc",
            ]
        ].copy()
        output["candidate_id"] = candidate_id
        output["model_family"] = candidate[
            "model_family"
        ]
        output["scope_type"] = candidate[
            "scope_type"
        ]
        output["selection_roles"] = candidate[
            "selection_roles"
        ]
        output["complexity_rank"] = int(
            candidate["complexity_rank"]
        )

        for column, values in zip(
            residual_columns,
            quantiles.T,
        ):
            output[column] = values

        hko_quantiles = (
            output[
                "forecast_daily_max_c"
            ].to_numpy(dtype=float)[:, None]
            + quantiles
        )
        for column, values in zip(
            hko_columns,
            hko_quantiles.T,
        ):
            output[column] = values

        output["predicted_residual_mean_c"] = (
            quantiles.mean(axis=1)
        )
        output[
            "predicted_residual_median_c"
        ] = quantiles[:, 49]
        output["predicted_hko_mean_c"] = (
            output["forecast_daily_max_c"]
            + output[
                "predicted_residual_mean_c"
            ]
        )
        output["predicted_hko_median_c"] = (
            output["forecast_daily_max_c"]
            + output[
                "predicted_residual_median_c"
            ]
        )
        output[
            "predictive_interval_80_lower_c"
        ] = hko_quantiles[:, 9]
        output[
            "predictive_interval_80_upper_c"
        ] = hko_quantiles[:, 89]
        output["prediction_status"] = "PASS"
        output["outcome_blind"] = True

        prediction_frames.append(output)
        diagnostic_rows.append(
            {
                "candidate_id": candidate_id,
                "model_family": candidate[
                    "model_family"
                ],
                "scope_id": scope_id,
                "selection_roles": candidate[
                    "selection_roles"
                ],
                "training_rows": len(train),
                "training_dates": train[
                    "event_date"
                ].nunique(),
                "latest_training_date": train[
                    "event_date"
                ].max(),
                "holdout_prediction_rows": output[
                    "evaluation_stage"
                ].eq(
                    "LOCKED_INTERNAL_HOLDOUT"
                ).sum(),
                "external_prediction_rows": output[
                    "evaluation_stage"
                ].eq(
                    "EXTERNAL_OOT_TRANSFER"
                ).sum(),
                "fit_status": "PASS",
                "fitted_parameters": json.dumps(
                    parameters
                ),
            }
        )

predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)
diagnostics = pd.DataFrame(
    diagnostic_rows
)

if len(predictions) != 3 * 159:
    raise AssertionError(
        f"Expected 477 blind prediction rows, "
        f"found {len(predictions)}."
    )

counts = predictions.groupby(
    "candidate_id"
).size()
if not counts.eq(159).all():
    raise AssertionError(
        f"Selected-candidate coverage differs:\n{counts}"
    )

holdout_counts = predictions.loc[
    predictions["evaluation_stage"].eq(
        "LOCKED_INTERNAL_HOLDOUT"
    )
].groupby("candidate_id").size()
external_counts = predictions.loc[
    predictions["evaluation_stage"].eq(
        "EXTERNAL_OOT_TRANSFER"
    )
].groupby("candidate_id").size()

if not holdout_counts.eq(40).all():
    raise AssertionError(
        f"Holdout coverage differs:\n{holdout_counts}"
    )
if not external_counts.eq(119).all():
    raise AssertionError(
        f"External coverage differs:\n{external_counts}"
    )

if predictions.duplicated(
    [
        "candidate_id",
        "event_date",
        "decision_rule",
    ]
).any():
    raise AssertionError(
        "Duplicate selected-candidate blind predictions."
    )

if not (
    np.diff(
        predictions[
            residual_columns
        ].to_numpy(dtype=float),
        axis=1,
    )
    >= -1e-12
).all():
    raise AssertionError(
        "Non-monotone blind predictive quantiles."
    )

forbidden_output = {
    "hko_daily_max_c",
    "residual_c",
    "forecast_error_c",
    "Y_event_int",
    "Y_no_int",
    "p_market",
    "market_binary_brier",
    "market_binary_log_score",
    "current_label_available_utc",
}

present = forbidden_output.intersection(
    predictions.columns
)
if present:
    raise AssertionError(
        "Blind predictions contain forbidden fields: "
        f"{sorted(present)}"
    )

if predictions[
    "refit_on_holdout_labels"
].any():
    raise AssertionError(
        "External predictions refit on holdout labels."
    )

print("Selected-model blind prediction generation: PASS")
print(f"Prediction rows: {len(predictions):,}")
display(diagnostics)

Selected-model blind prediction generation: PASS
Prediction rows: 477


/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99293/2543282381.py:413: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[column] = values
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99293/2543282381.py:413: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  output[column] = values
/var/folders/ck/dm6nl73d3v92cz_5d_bhjx_00000gn/T/ipykernel_99293/2543282381.py:413: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performan

,candidate_id,model_family,scope_id,selection_roles,training_rows,training_dates,latest_training_date,holdout_prediction_rows,external_prediction_rows,fit_status,fitted_parameters
0,catboost_quantile_pooled,TREE,pooled_all_rules,selected_tree,208,60,2026-05-19,40,119,PASS,"{""loss"": ""MultiQuantile"", ""training_quantiles""..."
1,gp_matern32_rule,GAUSSIAN_PROCESS,rule_specific_24h_prior,selected_gaussian_process,54,54,2026-05-19,10,30,PASS,"{""fitted_kernel"": ""0.407**2 * Matern(length_sc..."
2,gp_matern32_rule,GAUSSIAN_PROCESS,rule_specific_12h_prior,selected_gaussian_process,51,51,2026-05-19,10,30,PASS,"{""fitted_kernel"": ""0.455**2 * Matern(length_sc..."
3,gp_matern32_rule,GAUSSIAN_PROCESS,rule_specific_6h_prior,selected_gaussian_process,51,51,2026-05-20,10,29,PASS,"{""fitted_kernel"": ""0.161**2 * Matern(length_sc..."
4,gp_matern32_rule,GAUSSIAN_PROCESS,rule_specific_event_day_open,selected_gaussian_process,54,54,2026-05-20,10,30,PASS,"{""fitted_kernel"": ""0.1**2 * Matern(length_scal..."
5,pooled_empirical_residual,BASELINE,pooled_all_rules,selected_baseline|selected_overall,208,60,2026-05-19,40,119,PASS,"{""weighted_mean_residual_c"": 1.610833333333333..."


In [4]:
check_rows = []

def add_check(
    check: str,
    passed: bool,
    detail: str,
) -> None:
    check_rows.append(
        {
            "check": check,
            "passed": bool(passed),
            "detail": detail,
            "blocking": True,
        }
    )

add_check(
    "selected_candidates_3",
    len(unique_selected) == 3,
    unique_selected[
        "candidate_id"
    ].tolist().__str__(),
)
add_check(
    "blind_prediction_rows_477",
    len(predictions) == 477,
    f"rows={len(predictions)}",
)
add_check(
    "rows_per_selected_candidate_159",
    counts.eq(159).all(),
    counts.to_dict().__str__(),
)
add_check(
    "holdout_rows_per_candidate_40",
    holdout_counts.eq(40).all(),
    holdout_counts.to_dict().__str__(),
)
add_check(
    "external_rows_per_candidate_119",
    external_counts.eq(119).all(),
    external_counts.to_dict().__str__(),
)
add_check(
    "quantiles_monotone",
    (
        np.diff(
            predictions[
                residual_columns
            ].to_numpy(dtype=float),
            axis=1,
        )
        >= -1e-12
    ).all(),
    "99 residual quantiles",
)
add_check(
    "output_is_outcome_blind",
    not bool(
        forbidden_output.intersection(
            predictions.columns
        )
    ),
    "no realised outcome or market field",
)
add_check(
    "no_holdout_label_refit",
    not predictions[
        "refit_on_holdout_labels"
    ].any(),
    "identical pre-holdout fit transferred to June",
)
add_check(
    "all_prediction_status_pass",
    predictions[
        "prediction_status"
    ].eq("PASS").all(),
    "no selected-model fit failure",
)
add_check(
    "market_information_absent",
    "p_market" not in predictions.columns,
    "weather residual forecasts only",
)
add_check(
    "fixed_bridge_absent",
    not any(
        (
            "gaussian_bridge" in column.lower()
            or "ecmwf_proxy" in column.lower()
        )
        for column in predictions.columns
    ),
    "no archived bridge field",
)

integrity = pd.DataFrame(check_rows)
if not integrity["passed"].all():
    raise AssertionError(
        "18vD blocking checks failed:\n"
        + integrity.loc[
            ~integrity["passed"]
        ].to_string(index=False)
    )

issues = pd.DataFrame(
    columns=[
        "issue_level",
        "issue_code",
        "candidate_id",
        "scope_id",
        "evaluation_stage",
        "event_date",
        "decision_rule",
        "detail",
        "blocking",
    ]
)

print("18vD integrity checks: PASS")

18vD integrity checks: PASS


In [5]:
outputs = {
    "selected_registry": unique_selected,
    "predictions": predictions,
    "diagnostics": diagnostics,
    "integrity": integrity,
    "issues": issues,
}

paths = {
    "selected_registry": (
        OUT_DIR
        / "18vD_selected_candidate_registry.csv"
    ),
    "predictions": (
        OUT_DIR
        / "18vD_selected_candidate_blind_predictions.csv"
    ),
    "diagnostics": (
        OUT_DIR / "18vD_final_fit_diagnostics.csv"
    ),
    "integrity": (
        OUT_DIR / "18vD_integrity_checks.csv"
    ),
    "issues": OUT_DIR / "18vD_issues.csv",
}

for key, frame in outputs.items():
    output = frame.copy()

    for column in output.columns:
        if "date" in column.lower():
            if pd.api.types.is_datetime64_any_dtype(
                output[column]
            ):
                output[column] = output[
                    column
                ].dt.strftime("%Y-%m-%d")

        if (
            "freeze" in column.lower()
            or "cutoff" in column.lower()
            or column.lower().endswith("_utc")
            or "available" in column.lower()
            or "initialisation" in column.lower()
        ):
            output[column] = output[
                column
            ].astype("string")

    output.to_csv(
        paths[key],
        index=False,
    )

source_inventory = pd.DataFrame(
    [
        {
            "input_role": "18vA_candidate_registry",
            "path": str(
                CANDIDATE_PATH.relative_to(ROOT)
            ),
            "rows": len(candidates),
            "sha256": sha256_file(
                CANDIDATE_PATH
            ),
        },
        {
            "input_role": "18vA_quantile_grid",
            "path": str(
                QUANTILE_PATH.relative_to(ROOT)
            ),
            "rows": len(quantile_grid),
            "sha256": sha256_file(
                QUANTILE_PATH
            ),
        },
        {
            "input_role": "18vA_blind_feature_panel",
            "path": str(
                BLIND_FEATURE_PATH.relative_to(ROOT)
            ),
            "rows": len(blind),
            "sha256": sha256_file(
                BLIND_FEATURE_PATH
            ),
        },
        {
            "input_role": "18vC_selection",
            "path": str(
                SELECTION_PATH.relative_to(ROOT)
            ),
            "rows": 1,
            "sha256": sha256_file(
                SELECTION_PATH
            ),
        },
        {
            "input_role": "18vC_candidate_scores",
            "path": str(
                SCORE_PATH.relative_to(ROOT)
            ),
            "rows": len(scores),
            "sha256": sha256_file(
                SCORE_PATH
            ),
        },
        {
            "input_role": "18uB_final_fit_membership",
            "path": str(
                FINAL_FIT_PATH.relative_to(ROOT)
            ),
            "rows": len(final_fit),
            "sha256": sha256_file(
                FINAL_FIT_PATH
            ),
        },
        {
            "input_role": "18uB_evaluation_support",
            "path": str(
                EVALUATION_PATH.relative_to(ROOT)
            ),
            "rows": len(evaluation),
            "sha256": sha256_file(
                EVALUATION_PATH
            ),
        },
    ]
)
source_inventory_path = (
    OUT_DIR / "18vD_source_inventory.csv"
)
source_inventory.to_csv(
    source_inventory_path,
    index=False,
)

protocol = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "selected_candidates": (
        unique_selected[
            [
                "candidate_id",
                "model_family",
                "scope_type",
                "selection_roles",
            ]
        ].to_dict(orient="records")
    ),
    "internal_holdout_rows_per_candidate": 40,
    "external_rows_per_candidate": 119,
    "primary_external_protocol": (
        "identical pre-holdout fit transferred to June"
    ),
    "refit_on_holdout_labels": False,
    "holdout_or_external_outcomes_loaded": False,
    "market_information_used": False,
    "artificial_ensemble_features_used": False,
    "fixed_gaussian_bridge_used": False,
    "event_probability_mapping_pending": True,
    "holdout_and_external_scoring_pending": True,
}

protocol_path = OUT_DIR / "18vD_protocol.json"
protocol_path.write_text(
    json.dumps(
        protocol,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

summary = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "distinct_selected_candidates": int(
        len(unique_selected)
    ),
    "selected_baseline": selection[
        "selected_baseline"
    ],
    "selected_gaussian_process": selection[
        "selected_gaussian_process"
    ],
    "selected_tree": selection[
        "selected_tree"
    ],
    "selected_overall": selection[
        "selected_overall"
    ],
    "blind_prediction_rows": int(
        len(predictions)
    ),
    "rows_per_selected_candidate": 159,
    "internal_holdout_rows_per_candidate": 40,
    "external_rows_per_candidate": 119,
    "final_fit_diagnostic_rows": int(
        len(diagnostics)
    ),
    "holdout_or_external_outcomes_loaded": False,
    "refit_on_holdout_labels": False,
    "market_information_used": False,
    "artificial_ensemble_features_used": False,
    "fixed_gaussian_bridge_used": False,
    "event_probability_mapping_pending": True,
    "holdout_and_external_scoring_pending": True,
    "issue_rows": 0,
    "integrity_checks_passed": int(
        integrity["passed"].sum()
    ),
    "integrity_checks_total": int(
        len(integrity)
    ),
}

summary_path = OUT_DIR / "18vD_summary.json"
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scipy": __import__("scipy").__version__,
    "scikit_learn": __import__(
        "sklearn"
    ).__version__,
    "catboost": __import__("catboost").__version__,
    "revision": "v1",
}
environment_path = (
    OUT_DIR / "18vD_environment.json"
)
environment_path.write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

report_lines = [
    "# 18vD selected-model blind predictions",
    "",
    "**PASS**",
    "",
    "## Selected candidates",
    "",
    "| Candidate | Family | Scope | Roles |",
    "|---|---|---|---|",
]

for row in unique_selected.itertuples(index=False):
    report_lines.append(
        f"| {row.candidate_id} | "
        f"{row.model_family} | "
        f"{row.scope_type} | "
        f"{row.selection_roles} |"
    )

report_lines.extend(
    [
        "",
        "## Blind prediction support",
        "",
        (
            f"- Total candidate-date-rule predictions: "
            f"{len(predictions):,}"
        ),
        "- Internal holdout rows per candidate: 40",
        "- June external rows per candidate: 119",
        "",
        "## Leakage boundary",
        "",
        (
            "The output contains no realised HKO outcome, "
            "realised residual, contract outcome or market field."
        ),
        (
            "June uses the identical pre-holdout fit; there is "
            "no refit on internal-holdout labels."
        ),
        "",
        (
            "Contract-event probability mapping and holdout/June "
            "scoring remain pending."
        ),
    ]
)

report_path = (
    REPORT_DIR
    / "18vD_selected_models_blind_predictions_report.md"
)
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

manifest_rows = []
for root in [OUT_DIR, REPORT_DIR]:
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18vD_sha256_manifest.csv":
            continue

        manifest_rows.append(
            {
                "path": str(path.relative_to(ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = (
    OUT_DIR / "18vD_sha256_manifest.csv"
)
pd.DataFrame(manifest_rows).to_csv(
    manifest_path,
    index=False,
)

print(json.dumps(summary, indent=2))
print("18vD blind prediction release: PASS")

{
  "step": "18vD",
  "generated_at_utc": "2026-07-21T22:43:12.257304+00:00",
  "verdict": "PASS",
  "distinct_selected_candidates": 3,
  "selected_baseline": "pooled_empirical_residual",
  "selected_gaussian_process": "gp_matern32_rule",
  "selected_tree": "catboost_quantile_pooled",
  "selected_overall": "pooled_empirical_residual",
  "blind_prediction_rows": 477,
  "rows_per_selected_candidate": 159,
  "internal_holdout_rows_per_candidate": 40,
  "external_rows_per_candidate": 119,
  "final_fit_diagnostic_rows": 6,
  "holdout_or_external_outcomes_loaded": false,
  "refit_on_holdout_labels": false,
  "market_information_used": false,
  "artificial_ensemble_features_used": false,
  "fixed_gaussian_bridge_used": false,
  "event_probability_mapping_pending": true,
  "holdout_and_external_scoring_pending": true,
  "issue_rows": 0,
  "integrity_checks_passed": 11,
  "integrity_checks_total": 11
}
18vD blind prediction release: PASS
